In [17]:
import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report
)

In [18]:
# 1. 데이터 수집
titanic = sns.load_dataset("titanic")
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [19]:
# 2. 필요한 컬럼 선택

df = titanic[["survived","age","fare","sex","pclass"]].copy()
df

,survived,age,fare,sex,pclass
0,0,22.0,7.2500,male,3
1,1,38.0,71.2833,female,1
2,1,26.0,7.9250,female,3
3,1,35.0,53.1000,female,1
4,0,35.0,8.0500,male,3
...,...,...,...,...,...
886,0,27.0,13.0000,male,2
887,1,19.0,30.0000,female,1
888,0,NaN,23.4500,female,3
889,1,26.0,30.0000,male,1


In [20]:
# 3. 결측치 처리
# df.info()
df["age"] = df["age"].fillna(df["age"].median())
df

,survived,age,fare,sex,pclass
0,0,22.0,7.2500,male,3
1,1,38.0,71.2833,female,1
2,1,26.0,7.9250,female,3
3,1,35.0,53.1000,female,1
4,0,35.0,8.0500,male,3
...,...,...,...,...,...
886,0,27.0,13.0000,male,2
887,1,19.0,30.0000,female,1
888,0,28.0,23.4500,female,3
889,1,26.0,30.0000,male,1


In [21]:
# 4. 원 핫 인코딩

df = pd.get_dummies(df,columns=["sex"],drop_first=True) # drop_first=True 없애면 sex_male/sex_female 도출됨.
df["sex_male"] = df["sex_male"].astype(int)
df

,survived,age,fare,pclass,sex_male
0,0,22.0,7.2500,3,1
1,1,38.0,71.2833,1,0
2,1,26.0,7.9250,3,0
3,1,35.0,53.1000,1,0
4,0,35.0,8.0500,3,1
...,...,...,...,...,...
886,0,27.0,13.0000,2,1
887,1,19.0,30.0000,1,0
888,0,28.0,23.4500,3,0
889,1,26.0,30.0000,1,1


In [22]:
# 5. x,y 분리 (독립변수, 종속변수)

x = df[["age","fare","pclass","sex_male"]]
y = df["survived"]

In [23]:
# 6. 학습용 / 테스트용 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=100
)

In [24]:
# 7. 모델생성
model = LogisticRegression(max_iter=1000)

# 8. 학습
model.fit(X_train,y_train)

# 9. 예측
y_pred = model.predict(X_test)

# 10. 평가
print("정확도")
print(accuracy_score(y_test,y_pred))

print("분류리포트")
print(classification_report(y_test,y_pred))

정확도
0.7932960893854749
분류리포트
              precision    recall  f1-score   support

           0       0.80      0.87      0.83       104
           1       0.79      0.69      0.74        75

    accuracy                           0.79       179
   macro avg       0.79      0.78      0.78       179
weighted avg       0.79      0.79      0.79       179



In [25]:
# 11. 결과 지표
import statsmodels.api as sm 

# -------------------------
# 1. X, y 준비
# -------------------------
X = df[["age", "fare", "pclass", "sex_male"]]
y = df["survived"]

# -------------------------
# 2. 상수항 추가 (필수)
# -------------------------
X_sm = sm.add_constant(X)

# -------------------------
# 3. Logit 모델
# -------------------------
model = sm.Logit(y, X_sm)
result = model.fit()

# -------------------------
# 4. 결과 출력
# -------------------------
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.452022
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               survived   No. Observations:                  891
Model:                          Logit   Df Residuals:                      886
Method:                           MLE   Df Model:                            4
Date:                Thu, 18 Jun 2026   Pseudo R-squ.:                  0.3212
Time:                        11:41:19   Log-Likelihood:                -402.75
converged:                       True   LL-Null:                       -593.33
Covariance Type:            nonrobust   LLR p-value:                 3.282e-81
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.6553      0.509      9.153      0.000       3.659       5.652
age           -0.0331      0.